## Secrets, configs & placement

Two cluster concerns that build straight on earlier modules: getting sensitive files to the right containers, and getting tasks onto the right nodes.

**Swarm secrets** are module 08's secrets-as-files, promoted to a first-class cluster object stored **encrypted at rest** in the raft log:

```bash
echo "hunter2" | docker secret create db_password -
docker service create --secret db_password --name api myorg/api
```

Inside the container the value appears at **`/run/secrets/db_password`** (mode `0444`, root-owned), and — crucially — **only nodes running an `api` task ever receive it**; managers don't push secrets to nodes that don't need them. **Configs** are the identical mechanism for *non*-secret files (an `nginx.conf`), distributed and mounted the same way. Both are **immutable** — to change one you create a new object and update the service to use it.

**Placement** decides *where* tasks land, via three tools in `deploy: placement:`:

```yaml
deploy:
  replicas: 9
  placement:
    constraints: [ "node.labels.storage == ssd", "node.role == worker" ]
    preferences: [ { spread: node.labels.datacenter } ]
    max_replicas_per_node: 2
```

- **Constraints** are *hard* requirements — the DB only on SSD nodes, tasks only on workers, a given OS/arch (`docker node update --label-add storage=ssd nodeX` sets labels).
- **Preferences** are *soft* hints — spread evenly across datacenters.
- **`max_replicas_per_node`** caps how many replicas share a node, so one node's failure can't take them all (9 replicas capped at 2/node needs ≥5 eligible nodes).

Together these answer "this secret only where it's needed, this task only where it's allowed" — the fine-grained control that separates a toy cluster from a production one.